# Vanilla options: greeks, delta hedging, gamma trading

A vanilla call priced by Black–Scholes and Monte Carlo, its greeks, and what they mean in practice: delta hedging a
long option and the P&L of the hedged position, $\tfrac12\int\Gamma S^2(\sigma_{\text{realised}}^2 - \sigma_{\text{implied}}^2)\,dt$:
a bet on realised against implied volatility.
Docs: *Models and Pricing* (Black–Scholes, Delta hedging and gamma trading), *Exercises* (`equity.tex`).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq
from tensorquant.pricers.black import blackscholes_calc

## Market and option
TICKER1, 1Y call struck at the forward. Black–Scholes on the forward (here with the equivalent yield of the dividends:
same European price, a continuous carry for the hedging simulation).

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
EXPIRY = EVALUATION_DATE + timedelta(days=365)
F = float(eq.forward(EXPIRY))
K = round(F, 2)


def vanilla(option_type=tq.OptionType.Call, strike=K, expiry=EXPIRY):
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=option_type,
                            strike=strike, underlying=eq.ticker)


call = vanilla()
pricer = tq.BlackScholesPricer()                                    # continuous: equivalent dividend yield
pricer.price(call, mkt)
S0, T = float(eq.spot), float(call.time_to_maturity)
r, sigma_i = float(call.risk_free_rate), float(call.volatility)
q = r - np.log(F / S0) / T                                          # dividends + repo as one carry
print(f"S0 = {S0:.2f}  K = F = {K:.2f}  T = {T:.3f}  r = {r:.3%}  q = {q:.3%}  implied vol = {sigma_i:.2%}")
print(f"Black-Scholes call = {float(call.price):.4f}")

## Prices: Black–Scholes and Monte Carlo
GBM with the vol at $K$ reproduces Black–Scholes; local vol reprices the whole smile (the same vanilla, up to the
Monte Carlo and grid error). Put–call parity $C - P = P(0,T)(F - K)$.

In [ ]:
rows = [{"method": "Black-Scholes", "price": float(call.price), "s.e.": 0.0}]
for name, model, steps in (("MC GBM", tq.GeometricBrownianMotion, 1), ("MC local vol", tq.LocalVolatilityModel, 100)):
    o = vanilla()
    tq.VanillaMCPricer(model=model, n_paths=200_000, n_steps=steps, seed=3).price(o, mkt)
    rows.append({"method": name, "price": float(o.price), "s.e.": float(o.standard_error)})
put = vanilla(tq.OptionType.Put)
pricer.price(put, mkt)
DF = float(eq.discount_curve.discount(EXPIRY))
print(f"C - P = {float(call.price) - float(put.price):.10f}   P(0,T)(F - K) = {DF * (F - K):.10f}")
pd.DataFrame(rows).set_index("method").round(4)

## Greeks against the spot
By AAD (gamma: second tape), the vol held at the strike: delta $e^{-qT}N(d_1)$ from 0 to 1, gamma
$e^{-qT}n(d_1)/(S\sigma\sqrt T)$ and vega $S e^{-qT} n(d_1)\sqrt T$ peaked at the money, theta mostly negative (the price of
gamma). Close to expiry gamma concentrates on the strike, vega fades.

In [ ]:
spot_var, s_ref = eq.spot, float(eq.spot)


def call_greeks(expiry_days):
    out = []
    for x in np.linspace(0.6, 1.4, 81):
        spot_var.assign(x * K)
        o = vanilla(expiry=EVALUATION_DATE + timedelta(days=expiry_days))
        p = tq.BlackScholesPricer()
        with tf.GradientTape() as outer:
            p.price(o, mkt, autodiff=True)
            delta, vega, d_t = p.tape.gradient(o.price, [spot_var, eq.vol_surface.volatility_matrix, o.time_to_maturity])
        gamma = outer.gradient(delta, spot_var)
        out.append({"S/K": x, "price": float(o.price), "delta": float(delta), "gamma": float(gamma),
                    "vega": float(tf.reduce_sum(vega)) / 100, "theta": -float(d_t) / 365})   # per vol point, per day
    spot_var.assign(s_ref)
    return pd.DataFrame(out).set_index("S/K")


greeks_by_t = {f"{d // 30}M" if d < 365 else "1Y": call_greeks(d) for d in (30, 91, 365)}
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
for ax, g in zip(axes, ("price", "delta", "gamma", "vega", "theta")):
    for label, df in greeks_by_t.items():
        ax.plot(df.index, df[g], label=label)
    ax.axvline(1.0, color="grey", lw=0.8, ls=":")
    ax.set_title(g); ax.set_xlabel("S / K"); ax.grid(alpha=0.3)
axes[0].legend(); plt.tight_layout()

## Delta hedging
Buy the call at the implied vol, short $\Delta$ shares, rebalance at each step, cash at $r$, short stock paying the carry
$q$. Paths are GBM with a **realised** vol $\sigma_r$; the hedge uses the **implied** $\sigma_i$. The delta of every path at
every step comes from `blackscholes_calc` under a tape (vectorised AAD).

In [ ]:
def bs_price_delta(S, tau, sigma):
    S = tf.constant(S, tf.float64)
    with tf.GradientTape() as tape:
        tape.watch(S)
        price = blackscholes_calc(S, tf.constant(K, tf.float64), tf.constant(r, tf.float64),
                                  tf.constant(sigma, tf.float64), tf.constant(tau, tf.float64),
                                  tf.constant(q, tf.float64), tq.OptionType.Call)
    return price.numpy(), tape.gradient(price, S).numpy()


def hedge(sigma_r, n_steps=252, n_paths=4000, seed=1):
    # Delta-hedged long call: discounted P&L at expiry per path, and the paths
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    z = rng.standard_normal((n_paths, n_steps))
    S = np.empty((n_paths, n_steps + 1)); S[:, 0] = S0
    S[:, 1:] = S0 * np.exp(np.cumsum((r - q - 0.5 * sigma_r**2) * dt + sigma_r * np.sqrt(dt) * z, axis=1))
    price0, delta = bs_price_delta(np.full(n_paths, S0), T, sigma_i)
    cash = -price0 + delta * S0                                     # pay the call, short delta shares
    for i in range(1, n_steps + 1):
        cash = cash * np.exp(r * dt) - q * delta * S[:, i - 1] * dt  # interest; carry paid on the short stock
        if i < n_steps:
            tau = T - i * dt
            _, new_delta = bs_price_delta(S[:, i], tau, sigma_i)
            cash += (new_delta - delta) * S[:, i]                   # rebalance
            delta = new_delta
    pnl = np.maximum(S[:, -1] - K, 0.0) - delta * S[:, -1] + cash
    return pnl * np.exp(-r * T), S                                   # P&L in today's money


pnl, _ = hedge(sigma_i)
print(f"realised = implied: mean P&L {pnl.mean():.4f}, std {pnl.std():.4f}  (premium {float(call.price):.2f})")

With $\sigma_r = \sigma_i$ the hedge replicates the option: the P&L is zero on average, its noise is the discrete
rebalancing error, $\propto 1/\sqrt{n}$ for $n$ rebalances (half the error for four times the trades).

In [ ]:
rows = []
for n in (12, 52, 252, 1000):
    p, _ = hedge(sigma_i, n_steps=n, n_paths=2000)
    rows.append({"rebalances": n, "mean P&L": p.mean(), "std P&L": p.std(), "std x sqrt(n)": p.std() * np.sqrt(n)})
pd.DataFrame(rows).set_index("rebalances").round(4)

## Gamma trading: a bet on volatility
Over a step the hedged long option earns $\tfrac12\Gamma S^2\big((\Delta S/S)^2 - \sigma_i^2\,\Delta t\big)$: the gamma gain on the
realised move minus the theta paid at the implied vol. Summed to expiry,
$$\text{P\&L} \approx \tfrac12 \int_0^T \Gamma_t S_t^2\,(\sigma_r^2 - \sigma_i^2)\,dt :$$
**long gamma (long option, delta hedged) wins if the realised vol beats the implied** and loses otherwise; short gamma
the opposite. The weight $\Gamma S^2$ makes the P&L path dependent: vol realised near the strike counts most.

In [ ]:
rows, dist = [], {}
for shift in (-0.10, -0.05, 0.0, 0.05, 0.10):
    sigma_r = sigma_i + shift
    p, _ = hedge(sigma_r)
    dist[f"{sigma_r:.0%}"] = p
    rows.append({"realised vol": sigma_r, "implied vol": sigma_i, "mean P&L": p.mean(), "std P&L": p.std(),
                 "P(P&L > 0)": (p > 0).mean()})
pd.DataFrame(rows).set_index("realised vol").round(4)

In [ ]:
o = vanilla(); vp = tq.BlackScholesPricer(); vp.price(o, mkt, autodiff=True)
vega_1pt = float(tf.reduce_sum(vp.tape.gradient(o.price, eq.vol_surface.volatility_matrix))) / 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, p in dist.items():
    axes[0].hist(p, bins=60, alpha=0.5, label=f"realised {label}")
axes[0].set_title(f"Hedged long call P&L (implied {sigma_i:.0%})"); axes[0].legend(); axes[0].grid(alpha=0.3)
shifts = np.array([float(k.strip('%')) / 100 for k in dist]) - sigma_i
axes[1].plot(100 * shifts, [p.mean() for p in dist.values()], "o-", label="mean hedged P&L")
axes[1].plot(100 * shifts, vega_1pt * 100 * shifts, "--", label="vega x (realised - implied)")
axes[1].set_xlabel("realised - implied (vol points)"); axes[1].set_title("Long gamma = long realised vol")
axes[1].grid(alpha=0.3); axes[1].legend(); plt.tight_layout()

Path by path: the hedged P&L against the realised vol of each path (realised = implied on average here). Dispersion
around the line: the same realised vol is worth more if it happens near the strike, where $\Gamma S^2$ is large.

In [ ]:
p, S = hedge(sigma_i)
realised = np.std(np.diff(np.log(S), axis=1), axis=1) / np.sqrt(T / (S.shape[1] - 1))
plt.figure(figsize=(7, 4))
plt.scatter(100 * realised, p, s=4, alpha=0.4)
plt.axvline(100 * sigma_i, color="grey", ls=":")
plt.xlabel("realised vol of the path (%)"); plt.ylabel("hedged P&L"); plt.title("P&L vs realised vol, path by path")
plt.grid(alpha=0.3);

## Exercises
1. Repeat the hedge for a short call: what changes in the P&L distribution, and what is the trader betting on?
2. Hedge with the delta at the realised vol instead of the implied: does the mean P&L change? And its dispersion?
3. Take two paths with the same realised vol, one oscillating near the strike and one far from it: compare their P&L
   with the gamma formula.